In [ ]:
from microbit import display, sleep
import robotbit_library as r
import radio

radio.config(group=3)
radio.on()

msg = ""
cnt = 0

W, H = 5, 5

# Brightness map (0..9)
BRIGHT_OPEN   = 1
BRIGHT_WALL   = 5
BRIGHT_START  = 6
BRIGHT_END    = 8
BRIGHT_PATH   = 9

In [ ]:
def update_progress(cnt): #Copied from radio code, shows device looking for radio/
    if cnt < 50:
        if cnt % 10 == 0:
            x = cnt // 10
            display.set_pixel(x, 2, 9)
    else:
        cnt = 0
        display.clear()
    return cnt + 1

In [ ]:
def cut_string(str):
  str = input("Input radio string here\n")
  grid_text = [str[i:i+5] for i in range(0, len(str), 5)]

In [ ]:
def show_grid():
    """Render the base grid with barriers, open cells, S, and X."""
    for y in range(H):
        for x in range(W):
            if (x, y) == start:
                b = BRIGHT_START
            elif (x, y) == goal:
                b = BRIGHT_END
            elif grid[y][x] == 1:
                b = BRIGHT_WALL
            else:
                b = BRIGHT_OPEN
            display.set_pixel(x, y, b)

In [ ]:
def wavefront(g, start, goal):
    """BFS on 4-neighborhood. Returns (dist, path).
       dist[y][x] = steps from start; -1 if unreached.
       path: list of (x,y) from start -> goal (inclusive), or [] if none."""
    sx, sy = start
    gx, gy = goal
    if g[sy][sx] == 1 or g[gy][gx] == 1:
        return None, []

    dist = [[-1] * W for _ in range(H)]
    dist[sy][sx] = 0

    # Lightweight queue (two arrays + head index)
    qx, qy = [sx], [sy]
    head = 0

    while head < len(qx):
        x = qx[head]
        y = qy[head]
        head += 1

        if x == gx and y == gy:
            break

        # Explore 4-connected neighbors
        for nx, ny in ((x+1, y), (x-1, y), (x, y+1), (x, y-1)):
            if 0 <= nx < W and 0 <= ny < H:
                if g[ny][nx] == 0 and dist[ny][nx] == -1:
                    dist[ny][nx] = dist[y][x] + 1
                    qx.append(nx); qy.append(ny)

    # No path?
    if dist[gy][gx] == -1:
        return dist, []

    # Backtrack shortest path
    path = [(gx, gy)]
    x, y = gx, gy
    d = dist[y][x]
    while d > 0:
        # choose neighbor with d-1
        for nx, ny in ((x+1, y), (x-1, y), (x, y+1), (x, y-1)):
            if 0 <= nx < W and 0 <= ny < H and dist[ny][nx] == d - 1:
                path.append((nx, ny))
                x, y = nx, ny
                d -= 1
                break
    path.reverse()
    return dist, path

In [ ]:
def animate_path_step(path):
    """Draw the path once, step-by-step, leaving S and X as-is."""
    if not path:
        return
    # Skip the first and last entries (start and goal) for drawing
    for i, (x, y) in enumerate(path):
        if (x, y) != start and (x, y) != goal:
            display.set_pixel(x, y, BRIGHT_PATH)
            sleep(130)

In [ ]:
def blink_path(path, times=2):
    """Blink the final path over the base grid, for visibility."""
    if not path:
        return

    for _ in range(times):
        # Turn on path pixels
        show_path(path)
        sleep(350)
        # Restore base grid
        show_grid()
        sleep(250)

In [ ]:
def show_path(path):
    for (x, y) in path:
        if (x, y) != start and (x, y) != goal:
            display.set_pixel(x, y, BRIGHT_PATH)

In [ ]:
def checkExpectedness(grid, threshold, pin):
    sensorValue = read_sensor(pin) # FIND ACTUAL FUNCTION

    if sensorValue > threshold:
        nextState = 1 # wall
    else:
        nextState = 0 # open space

    # Get the expected state
    expectedValue = grid[targetCell[1]][targetCell[0]]

    if nextState != expectedValue: # Unexpected wall detected
        # Update the grid
        grid[targetCell[1]][targetCell[0]] = 1
        return True
    else:
        return False

In [ ]:
def determineTurn(currentIndex):
    # THIS WAS ALREADY WRITTEN

In [ ]:
def turnLeft():
    r.motor(1, 100)
    r.motor(2, -100)
    sleep(370)
    r.motor(1,0)
    r.motor(2,0)
    sleep(500)

In [ ]:
def turnRight():
    r.motor(1, -100)
    r.motor(2, 100)
    sleep(355)
    r.motor(1,0)
    r.motor(2,0)
    sleep(500)

In [ ]:
def move(length):
    for i in range(length):
        r.motor(1, -100)
        r.motor(2, -83)
        sleep(1000)
        r.motor(1,0)
        r.motor(2,0)
        sleep(500)

In [ ]:
# ------------------------ Main flow ------------------------

# Wait for radio map
while len(msg) != 25:
    incoming = radio.receive()

    if incoming and len(incoming) == 25:
        msg = incoming
        radio.send(msg)
        print(msg)
    else:
        cnt = update_progress(cnt)

    sleep(250)

# Convert radio string into 5x5 grid
grid_text = [msg[i:i+5] for i in range(0, 25, 5)]

# Parse the grid
grid = [[0] * W for _ in range(H)]
start = None
goal = None

for y, row in enumerate(grid_text):
    for x, ch in enumerate(row):
        if ch == '1':
            grid[y][x] = 1
        elif ch == '0':
            grid[y][x] = 0
        elif ch == 'S':
            start = (x, y)
        elif ch == 'X':
            goal = (x, y)

show_grid()
sleep(700)

r.setup()

dist, path = wavefront(grid, start, goal)

if not path:
    display.scroll("NO PATH")
    show_grid()
else:
    animate_path_step(path)
    blink_path(path, times=2)
    show_path(path)

In [ ]:
currentPos = (start[0], start[1], 180)
for target in path[1:]:
    if target[1] < currentPos[1]:
        targetAngle = 0
    elif target[1] > currentPos[1]:
        targetAngle = 180
    elif target[0] < currentPos[0]:
        targetAngle = 90
    else:
        targetAngle = -90

    diff = (targetAngle - currentPos[2] + 180) % 360 - 180

    if diff == 180:
        turnRight()
        turnRight()
    elif diff < 0:
        turnRight()
    elif diff > 0:
        turnLeft()

    move(1)

    currentPos = (target[0], target[1], targetAngle)